## 10.1 CNN综合案例 - 数据准备

#### 1. 案例目标与本节任务

##### 1.1 这一章我们要做什么
从这一章开始，我们不再只学单个知识点，

而是要做一个 CNN 图像分类综合案例。📸

这个案例会把我们之前学过的内容串起来，包括：
* 使用 ImageFolder 读取图像数据
* 使用 transforms 做图像预处理和数据增强
* 使用 DataLoader 组成 batch
* 构建 CNN 网络
* 配置损失函数、优化器、学习率
* 使用 scheduler 学习率衰减
* 在分类头中加入 Dropout
* 完成训练、验证与测试

也就是说，这将是一个真正完整的 CNN 实战流程。

##### 1.2 本小节只做什么
这一小节我们先只做第一步：

数据准备（Data Preparation）

这一部分的目标是先把数据相关的基础打好，包括：
* 认识数据集
* 理解目录结构
* 明确训练集、验证集、测试集的来源
* 为后面使用 ImageFolder 和 DataLoader 做准备

所以这一节还不会正式开始写模型训练代码，

而是先把“数据从哪里来、长什么样、怎么组织”弄清楚。

#### 2. 为什么 CNN 案例要先从数据准备开始

##### 2.1 深度学习的第一步永远是数据
无论是 MLP、CNN，还是后面更复杂的模型，

深度学习项目的第一步几乎永远都是：

>先准备好数据。

因为如果数据没有整理清楚，后面的这些内容都会受到影响：
* 模型输入 shape 是否正确
* 标签是否对应正确
* 图像是否完成预处理
* 训练集和验证集是否合理分开
* batch 是否能正常组成

所以数据准备不是“前置小事”，

而是整个案例能不能顺利跑起来的基础。🧱

##### 2.2 CNN 对数据格式要求更明确
在图像任务中，数据不像表格那样天然整齐。

图像数据通常涉及：
* 文件夹结构
* 图像尺寸不一致
* 通道数问题（RGB / 灰度）
* 标签来自文件夹名称
* 训练和测试目录分开

所以在 CNN 中，

数据准备通常比普通表格任务更重要，也更具体。

#### 3. 图像分类数据集的典型目录结构

##### 3.1 ImageFolder 最适合的数据结构
ImageFolder 最常见、最标准的数据结构是下面这种：
```
dataset/
    train/
        class_1/
            img1.jpg
            img2.jpg
            ...
        class_2/
            img1.jpg
            img2.jpg
            ...
        class_3/
            ...
    test/
        class_1/
            img1.jpg
            ...
        class_2/
            img1.jpg
            ...
        class_3/
            ...
```

它的核心规律就是：

上一级目录表示数据集划分，下一层目录名表示类别名。

##### 3.2 这个结构为什么特别适合 PyTorch
因为 ImageFolder 会自动完成两件事：

**（1）自动读取图片**

它会遍历每个类别文件夹中的图像文件。

**（2）自动生成标签**

它会根据文件夹名称自动建立类别索引，例如：

`{'buildings': 0, 'forest': 1, 'glacier': 2, ...}`

也就是说，

我们不需要自己单独再写一个标签表格。

只要目录结构规范，ImageFolder 就能自动帮我们处理很多事情。✅

#### 4. 训练集、验证集、测试集怎么理解

##### 4.1 训练集（Training Set）
训练集是模型真正“学习”的数据。

在训练过程中，模型会不断看到训练集中的图像，并根据误差更新参数。

所以训练集的作用是：

让模型学习输入图像和类别之间的映射关系。

##### 4.2 验证集（Validation Set）
验证集不是拿来更新参数的，

而是用来在训练过程中观察模型表现的。

它的主要作用包括：
* 监控模型是否过拟合
* 比较不同 epoch 的效果
* 为 early stopping 或 scheduler 提供参考
* 帮助我们调参

所以验证集更像是：

训练过程中的“阶段性检查数据” 🔍

##### 4.3 测试集（Test Set）
测试集通常放在训练全部结束之后再使用。

它的作用是评估模型在“没见过的新数据”上的最终表现。

所以测试集更像是：

最后验收模型泛化能力的数据。

#### 5. 本案例使用的数据集 - CIFAR-10

##### 5.1 为什么我们选择 CIFAR-10
这次综合案例，我们直接使用 CIFAR-10。

它是图像分类中最经典的数据集之一，非常适合用来做 CNN 的完整入门案例。

CIFAR-10 一共包含 60,000 张彩色图片，分为 10 个类别，其中 50,000 张训练图像、10,000 张测试图像；每张图片大小都是 32 × 32。 

它适合我们当前阶段，主要有几个原因：
* 数据集经典，很多 CNN 教学案例都会用它  
* 图片尺寸较小，训练成本相对低
* 是标准的多分类任务
* 很适合练习完整流程：数据读取、预处理、CNN 搭建、训练、验证、测试

##### 5.2 CIFAR-10 的 10 个类别
CIFAR-10 包含以下 10 个类别：
* airplane
* automobile
* bird
* cat
* deer
* dog
* frog
* horse
* ship
* truck  

这说明我们的案例最终是一个：

10 分类图像识别任务

也就是说，模型最后输出层通常需要有：

10 个输出神经元

分别对应这 10 个类别。

##### 5.3 CIFAR-10 的图像特点
CIFAR-10 中的每张图片都是：
* 彩色图像
* 3 个通道（RGB）
* 32 × 32 像素

所以在 PyTorch 中，一张图片送入模型后的常见形状就是：

`3 × 32 × 32`

这也意味着，后面我们在构建 CNN 时，第一层卷积通常要写成：

`nn.Conv2d(in_channels=3, ...)`

因为输入通道数不是 1，而是 3。

#### 6. 使用 CIFAR-10 时需要先明确的一点

##### 6.1 这次不再使用 ImageFolder 作为主读取方式
这里要特别注意：

我们原本前面设想的综合案例，想重点练习的是 `ImageFolder`。

但如果这次正式改用 CIFAR-10，那么更标准、也更常见的读取方式就不是 `ImageFolder`，而是直接使用：

`torchvision.datasets.CIFAR10`

因为 `torchvision` 已经为 CIFAR-10 提供了现成的数据集接口，并且支持：
* 自动下载
* 自动区分训练集和测试集
* 直接配合 transform
* 直接交给 DataLoader 使用

##### 6.2 这意味着什么
这意味着这次案例中：
* 数据读取主线：我们会用 torchvision.datasets.CIFAR10
* 图像预处理：仍然使用 transforms
* batch 组织：仍然使用 DataLoader

##### 6.3 如果后面想使用 ImageFolder
把 CIFAR-10 导出并整理成按类别文件夹存放的结构，再用 `ImageFolder` 重新读取。

但从“标准 PyTorch CIFAR-10 流程”来说，

这次主线更推荐直接使用：

`torchvision.datasets.CIFAR10`

#### 7. 本案例的数据准备思路

##### 7.1 训练集和测试集的来源
CIFAR-10 官方已经提供了固定的：
* 训练集：50,000 张
* 测试集：10,000 张  

所以这次我们不需要自己再额外手动拆出 test 集。

##### 7.2 验证集怎么来
不过，CIFAR-10 的标准接口通常只直接给我们：
* train
* test

并不会额外单独给出 validation set。 

所以在这个综合案例中，我们通常会这样做：
* 先读取完整训练集
* 再从训练集中划分出一部分作为验证集

例如常见做法是：
* 训练集：80%
* 验证集：20%

这样后面训练时我们就有：
* train：真正训练参数
* val：观察泛化、调参、配合 scheduler
* test：最终评估